In [8]:
# ==============================================================================
# 1. IMPORTS
# ==============================================================================

# OpenPIV
from openpiv import windef
from openpiv import tools, scaling, validation, filters, preprocess
import openpiv.pyprocess as process
from openpiv import pyprocess

# General
import numpy as np
import tifffile as tif
import matplotlib.pyplot as plt
import pandas as pd
from roifile import ImagejRoi

import pathlib
from time import time
import warnings
%matplotlib qt

# Custom functions
from src.PIV import run_PIV_on_frames

In [16]:
# ==============================================================================
# 4. REFERENCE FRAME COMPARISON
# ==============================================================================

# SECTION Standard unstressed reference frame 

# Load data 
data_path = '/mnt/crunch/Clark/Fly_TFM/data/first/reference=left_the_frame/' 
filename = 'PIVlab_0350.txt' 

data = pd.read_csv(data_path + filename, skiprows=2)

# Get column values 
x_piv = data['x [px]'].values
y_piv = data['y [px]'].values
u_piv = data['u [px/frame]'].values
v_piv = data['v [px/frame]'].values

print(x_piv.shape, y_piv.shape, u_piv.shape, v_piv.shape)

# Load manual centerline for now
roi = ImagejRoi.fromfile('/mnt/crunch/Clark/Fly_TFM/data/first/0001-1216-0912.roi')
centerline_points = roi.coordinates() # numpy array of (x, y)

# Visualize 
plt.figure(figsize=(10, 10))
plt.quiver(x_piv, y_piv, u_piv, v_piv)
plt.plot(centerline_points[:, 0], centerline_points[:, 1], 'r-o')
plt.axis('equal')
points = plt.ginput(n=-1, timeout=0) 
plt.close()

(3969,) (3969,) (3969,) (3969,)


In [ ]:
from matplotlib.path import Path as MplPath

polygon = MplPath(points)
inside = polygon.contains_points(np.column_stack([x_piv, y_piv]))

near_larva_mask = inside
far_field_mask = ~inside

# Visualize gridpoints that were kept vs excluded from my terrible clicking
plt.figure(figsize=(10, 10))
plt.scatter(x_piv[inside], y_piv[inside], c='red', s=10, label='near (excluded)')
plt.scatter(x_piv[~inside], y_piv[~inside], c='blue', s=10, label='far (kept)')
plt.axis('equal')
plt.legend()
plt.show()

# Visualize only the backgroun signal vectors (far field)
plt.figure(figsize=(10,10))
plt.quiver(x_piv[~inside], y_piv[~inside], u_piv[~inside], v_piv[~inside])
plt.axis('equal')
plt.show()

In [ ]:
# Fit the data

def design_matrix(x, y):
    return np.column_stack([np.ones_like(x), x, y, x**2, y**2, x*y])

A_far = design_matrix(x_piv[~inside], y_piv[~inside])

coeffs_u, *_ = np.linalg.lstsq(A_far, u_piv[~inside], rcond=None)
coeffs_v, *_ = np.linalg.lstsq(A_far, v_piv[~inside], rcond=None)

# evaluate the fit everywhere, including inside the excluded blob
A_all = design_matrix(x_piv, y_piv)
u_fit = A_all @ coeffs_u
v_fit = A_all @ coeffs_v

In [ ]:
# Evaluate goodness of fit

u_fit_inside = u_fit[inside]
u_actual_inside = u_piv[inside]

v_fit_inside = v_fit[inside]
v_actual_inside = v_piv[inside]

same_sign_u = np.sign(u_fit_inside) == np.sign(u_actual_inside)
same_sign_v = np.sign(v_fit_inside) == np.sign(v_actual_inside)

print(f"u: {same_sign_u.mean()*100:.0f}% of near-larva points share sign with the fit")
print(f"v: {same_sign_v.mean()*100:.0f}% of near-larva points share sign with the fit")

u: 43% of near-larva points share sign with the fit
v: 45% of near-larva points share sign with the fit


In [24]:
# Visualize the 3d surface data (for u, x-component)

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Actual far-field data points
ax.scatter(x_piv[~inside], y_piv[~inside], u_piv[~inside], c='blue', s=5, label='actual (far-field)')

# Fitted surface, evaluated on a smooth grid for a clean mesh
xx, yy = np.meshgrid(
    np.linspace(x_piv.min(), x_piv.max(), 50),
    np.linspace(y_piv.min(), y_piv.max(), 50)
)
zz = (coeffs_u[0] + coeffs_u[1]*xx + coeffs_u[2]*yy 
    + coeffs_u[3]*xx**2 + coeffs_u[4]*yy**2 + coeffs_u[5]*xx*yy)

ax.plot_surface(xx, yy, zz, alpha=0.3, color='orange')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('u')
plt.show()

In [23]:
# Visualize the 3d surface data (for v, y-component)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(x_piv[~inside], y_piv[~inside], v_piv[~inside], c='blue', s=5, label='actual (far-field)')

zz_v = (coeffs_v[0] + coeffs_v[1]*xx + coeffs_v[2]*yy 
        + coeffs_v[3]*xx**2 + coeffs_v[4]*yy**2 + coeffs_v[5]*xx*yy)

ax.plot_surface(xx, yy, zz_v, alpha=0.3, color='orange')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('v')
plt.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=x_piv[~inside], y=y_piv[~inside], z=u_piv[~inside],
    mode='markers', marker=dict(size=2, color='blue'), name='actual (far-field)'
))

fig.add_trace(go.Surface(
    x=xx, y=yy, z=zz, opacity=0.4, colorscale='Oranges', showscale=False, name='fit'
))

fig.update_layout(scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='u'))
fig.write_html('u_surface_check.html')

In [26]:
fig_v = go.Figure()

fig_v.add_trace(go.Scatter3d(
    x=x_piv[~inside], y=y_piv[~inside], z=v_piv[~inside],
    mode='markers', marker=dict(size=2, color='blue'), name='actual (far-field)'
))

fig_v.add_trace(go.Surface(
    x=xx, y=yy, z=zz_v, opacity=0.4, colorscale='Oranges', showscale=False, name='fit'
))

fig_v.update_layout(scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='v'))
fig_v.write_html('v_surface_check.html')

In [43]:
u_residual = u_piv - u_fit
v_residual = v_piv - v_fit

# Visualize only the backgroun signal vectors (far field)
plt.figure(figsize=(10,10))
plt.quiver(x_piv[~inside], y_piv[~inside], u_piv[~inside], v_piv[~inside])
plt.axis('equal')
plt.show()

plt.figure(figsize=(10,10))
plt.quiver(x_piv[~inside], y_piv[~inside], u_residual[~inside], v_residual[~inside])
plt.axis('equal')
plt.show()


# Print average vector magnitude in far field and near field region of actual data
print(f"Average vector magnitude in far field (actual): {np.sqrt(u_piv[~inside]**2 + v_piv[~inside]**2).mean():.4f}")
print(f"Average vector magnitude in near field (actual): {np.sqrt(u_piv[inside]**2 + v_piv[inside]**2).mean():.4f}")

# Print average vector magnitude in far field and near field region of residual
print(f"Average vector magnitude in far field (residual): {np.sqrt(u_residual[~inside]**2 + v_residual[~inside]**2).mean():.4f}")
print(f"Average vector magnitude in near field (residual): {np.sqrt(u_residual[inside]**2 + v_residual[inside]**2).mean():.4f}")

Average vector magnitude in far field (actual): 2.6441
Average vector magnitude in near field (actual): 4.0594
Average vector magnitude in far field (residual): 1.0311
Average vector magnitude in near field (residual): 4.0150


In [38]:
u_residual2 = u_residual.copy()
v_residual2 = v_residual.copy()

u_residual2[inside] = u_residual[inside] - u_piv[inside]
v_residual2[inside] = v_residual[inside] - v_piv[inside]

plt.figure(figsize=(10,10))
plt.quiver(x_piv, y_piv, u_residual2, v_residual2)
plt.axis('equal')
plt.show()